In [1]:
# # Final Dataset Generation for Satellite Image Super-Resolution

# Objective:
# Generate a complete LR–HR dataset from Sentinel-2 Level-2A imagery
# by performing RGB creation, normalization, patch extraction, and
# LR simulation, and saving the data in train/validation/test format.

In [1]:
import os
import rasterio
import numpy as np
import cv2
from glob import glob
from sklearn.model_selection import train_test_split


In [2]:
# Root project path
PROJECT_ROOT = r"A:\Main Project\Satellite_SR"

# Where SAFE files are stored
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

# Output dataset path
DATASET_ROOT = os.path.join(PROJECT_ROOT, "dataset")

# Create dataset folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(DATASET_ROOT, split, "HR"), exist_ok=True)
    os.makedirs(os.path.join(DATASET_ROOT, split, "LR"), exist_ok=True)

print("Dataset folders created successfully")


Dataset folders created successfully


In [34]:
def read_rgb_from_safe(safe_path):
    # Select correct granule (with R10m data)
    granules = glob(os.path.join(safe_path, "GRANULE", "*"))
    granule_path = None
    for g in granules:
        if os.path.exists(os.path.join(g, "IMG_DATA", "R10m")):
            granule_path = g
            break

    if granule_path is None:
        raise RuntimeError("No valid GRANULE with R10m found")

    r10m_path = os.path.join(granule_path, "IMG_DATA", "R10m")

    b02_path = glob(os.path.join(r10m_path, "*_B02_10m.jp2"))[0]
    b03_path = glob(os.path.join(r10m_path, "*_B03_10m.jp2"))[0]
    b04_path = glob(os.path.join(r10m_path, "*_B04_10m.jp2"))[0]

    with rasterio.open(b02_path) as src:
        B02 = src.read(1)
    with rasterio.open(b03_path) as src:
        B03 = src.read(1)
    with rasterio.open(b04_path) as src:
        B04 = src.read(1)

    rgb = np.stack([B04, B03, B02], axis=-1).astype("float32")
    return rgb


In [35]:
def extract_patches(image, patch_size=96, stride=96):
    patches = []
    h, w, _ = image.shape

    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patch = image[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)

    return patches


In [36]:
def normalize_patch(patch):
    max_val = patch.max()
    if max_val > 0:
        return patch / max_val
    else:
        return patch


In [37]:
def generate_lr_patch(hr_patch, scale=2):
    h, w, _ = hr_patch.shape

    lr = cv2.resize(
        hr_patch,
        (w // scale, h // scale),
        interpolation=cv2.INTER_AREA
    )

    lr_up = cv2.resize(
        lr,
        (w, h),
        interpolation=cv2.INTER_CUBIC
    )

    return lr_up


In [38]:
DATA_ROOT = r"A:\Main Project\Satellite_SR\data"

safe_files = [
    p for p in glob(os.path.join(DATA_ROOT, "**", "*.SAFE"), recursive=True)
    if os.path.isdir(p)
]

print("Number of SAFE directories:", len(safe_files))
for s in safe_files:
    print(os.path.basename(s))


Number of SAFE directories: 4
S2B_MSIL2A_20251213T051119_N0511_R019_T43PGQ_20251213T071206.SAFE
S2B_MSIL2A_20251223T051129_N0511_R019_T43PGQ_20251223T084635.SAFE
S2B_MSIL2A_20250325T045659_N0511_R119_T44PLV_20250325T074633.SAFE
S2B_MSIL2A_20250504T045659_N0511_R119_T44PLV_20250504T072305.SAFE


In [42]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cv2
import os
from glob import glob

DATASET_ROOT = r"A:\Main Project\Satellite_SR\dataset"
PATCH_SIZE = 96
SCALE = 2

patch_counter = 0

for safe_idx, safe_path in enumerate(safe_files):
    print(f"\nProcessing SAFE {safe_idx+1}/{len(safe_files)}:")
    print(os.path.basename(safe_path))

    # Find correct granule
    granules = glob(os.path.join(safe_path, "GRANULE", "*"))
    granule_path = None
    for g in granules:
        if os.path.exists(os.path.join(g, "IMG_DATA", "R10m")):
            granule_path = g
            break

    r10m = os.path.join(granule_path, "IMG_DATA", "R10m")

    b02_path = glob(os.path.join(r10m, "*_B02_10m.jp2"))[0]
    b03_path = glob(os.path.join(r10m, "*_B03_10m.jp2"))[0]
    b04_path = glob(os.path.join(r10m, "*_B04_10m.jp2"))[0]

    with rasterio.open(b02_path) as src_b02, \
         rasterio.open(b03_path) as src_b03, \
         rasterio.open(b04_path) as src_b04:

        height, width = src_b02.height, src_b02.width

        for row in range(0, height - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, width - PATCH_SIZE + 1, PATCH_SIZE):

                window = Window(col, row, PATCH_SIZE, PATCH_SIZE)

                try:
                    B02 = src_b02.read(1, window=window)
                    B03 = src_b03.read(1, window=window)
                    B04 = src_b04.read(1, window=window)
                except Exception as e:
                    # Skip problematic window
                    continue


                # Skip empty patches
                if B04.max() == 0:
                    continue

                # Stack RGB
                hr = np.stack([B04, B03, B02], axis=-1).astype("float32")

                # Normalize per patch
                hr /= hr.max()

                # Generate LR
                lr = cv2.resize(
                    hr,
                    (PATCH_SIZE // SCALE, PATCH_SIZE // SCALE),
                    interpolation=cv2.INTER_AREA
                )
                lr = cv2.resize(
                    lr,
                    (PATCH_SIZE, PATCH_SIZE),
                    interpolation=cv2.INTER_CUBIC
                )

                # Train / Val / Test split
                r = np.random.rand()
                if r < 0.7:
                    split = "train"
                elif r < 0.85:
                    split = "val"
                else:
                    split = "test"

                hr_path = os.path.join(DATASET_ROOT, split, "HR", f"img_{patch_counter:06d}.png")
                lr_path = os.path.join(DATASET_ROOT, split, "LR", f"img_{patch_counter:06d}.png")

                cv2.imwrite(hr_path, (hr * 255).astype(np.uint8))
                cv2.imwrite(lr_path, (lr * 255).astype(np.uint8))

                patch_counter += 1

    print(f"  Patches saved so far: {patch_counter}")

print("\n✅ DATASET CREATION COMPLETED")
print("Total patches saved:", patch_counter)



Processing SAFE 1/4:
S2B_MSIL2A_20251213T051119_N0511_R019_T43PGQ_20251213T071206.SAFE
  Patches saved so far: 12288

Processing SAFE 2/4:
S2B_MSIL2A_20251223T051129_N0511_R019_T43PGQ_20251223T084635.SAFE
  Patches saved so far: 24692

Processing SAFE 3/4:
S2B_MSIL2A_20250325T045659_N0511_R119_T44PLV_20250325T074633.SAFE
  Patches saved so far: 35598

Processing SAFE 4/4:
S2B_MSIL2A_20250504T045659_N0511_R119_T44PLV_20250504T072305.SAFE
  Patches saved so far: 46550

✅ DATASET CREATION COMPLETED
Total patches saved: 46550
